In [13]:
from config import DATASET_ROOT, LOG_DIR, PROCESSED_DIR, VARIABLES
from climate_preprocessing.grd import discover_year_files
from climate_preprocessing.logging_utils import configure_logging
from climate_preprocessing.tensor import create_stacked_tensors

logger = configure_logging(LOG_DIR)
files = {key: {item.year: item for item in discover_year_files(DATASET_ROOT, variable)} for key, variable in VARIABLES.items()}
years = sorted(set.intersection(*(set(value) for value in files.values())))
create_stacked_tensors(files, years, PROCESSED_DIR, logger)

ModuleNotFoundError: No module named 'config'

In [ ]:
import numpy as np
from pathlib import Path

# =====================================================
# Configuration
# =====================================================

INPUT_DIR = Path("E:\Projects\ISRO_BAH_2026\processed\stacked_tensor")

TRAIN_FILE = INPUT_DIR / "train_tensor_2012_2024.npz"
VALID_FILE = INPUT_DIR / "validation_2025.npz"

OUTPUT_TRAIN = INPUT_DIR / "train_tensor_kerala.npz"
OUTPUT_VALID = INPUT_DIR / "validation_tensor_kerala.npz"

LAT_MIN = 8.0
LAT_MAX = 13.0

LON_MIN = 74.5
LON_MAX = 77.5

# =====================================================
# Crop Function
# =====================================================

def crop_tensor(npz_file, output_file):

    data = np.load(npz_file)

    tensor = data["tensor"]
    dates = data["dates"]
    channels = data["channels"]
    lat = data["latitudes"]
    lon = data["longitudes"]
    mask = data["rainfall_mask"]

    lat_idx = np.where((lat >= LAT_MIN) & (lat <= LAT_MAX))[0]
    lon_idx = np.where((lon >= LON_MIN) & (lon <= LON_MAX))[0]

    tensor_crop = tensor[
        :,
        :,
        lat_idx.min():lat_idx.max()+1,
        lon_idx.min():lon_idx.max()+1
    ]

    mask_crop = mask[
        lat_idx.min():lat_idx.max()+1,
        lon_idx.min():lon_idx.max()+1
    ]

    lat_crop = lat[
        lat_idx.min():lat_idx.max()+1
    ]

    lon_crop = lon[
        lon_idx.min():lon_idx.max()+1
    ]

    np.savez_compressed(
        output_file,
        tensor=tensor_crop,
        dates=dates,
        channels=channels,
        latitudes=lat_crop,
        longitudes=lon_crop,
        rainfall_mask=mask_crop,
    )

    print("="*60)
    print(output_file.name)
    print("="*60)
    print("Tensor Shape :", tensor_crop.shape)
    print("Latitude     :", lat_crop[0], "→", lat_crop[-1])
    print("Longitude    :", lon_crop[0], "→", lon_crop[-1])
    print("Valid Cells  :", mask_crop.sum())
    print()

# =====================================================
# Execute
# =====================================================

crop_tensor(TRAIN_FILE, OUTPUT_TRAIN)
crop_tensor(VALID_FILE, OUTPUT_VALID)

print("Done.")

<>:8: SyntaxWarning: invalid escape sequence '\P'
<>:8: SyntaxWarning: invalid escape sequence '\P'
C:\Users\Aditya Raj\AppData\Local\Temp\ipykernel_1284\1598298939.py:8: SyntaxWarning: invalid escape sequence '\P'
  INPUT_DIR = Path("E:\Projects\ISRO_BAH_2026\processed\stacked_tensor")


train_tensor_kerala.npz
Tensor Shape : (4749, 3, 21, 13)
Latitude     : 8.0 → 13.0
Longitude    : 74.5 → 77.5
Valid Cells  : 156

validation_tensor_kerala.npz
Tensor Shape : (365, 3, 21, 13)
Latitude     : 8.0 → 13.0
Longitude    : 74.5 → 77.5
Valid Cells  : 156

Done.
